# Notebook 03 — Model Training & Evaluation
**Credit Card Approval Prediction System**

This notebook covers:
- Label encoding of categorical columns
- Train/test split (80/20)
- Training all 4 classifiers
- Confusion matrix visualization for each
- F1-score comparison table
- `RandomizedSearchCV` hyperparameter tuning
- Saving `model.pkl` and `encoders.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, sys, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier

print('Libraries loaded ✓')

## 1. Reload & Re-run Preprocessing (quick)

In [ ]:
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from train import load_data, data_cleaning

app, credit = load_data()
merged = data_cleaning(app, credit)
print(f'Merged shape: {merged.shape}')

## 2. Step 3.6 — Label Encoding

In [ ]:
gender_le  = LabelEncoder()
car_le     = LabelEncoder()
realty_le  = LabelEncoder()

credit_app = merged.copy()
credit_app['CODE_GENDER']    = gender_le.fit_transform(credit_app['CODE_GENDER'])
credit_app['FLAG_OWN_CAR']   = car_le.fit_transform(credit_app['FLAG_OWN_CAR'])
credit_app['FLAG_OWN_REALTY']= realty_le.fit_transform(credit_app['FLAG_OWN_REALTY'])

print('Label encoders fitted:')
print(f'  gender  classes : {list(gender_le.classes_)}')
print(f'  car     classes : {list(car_le.classes_)}')
print(f'  realty  classes : {list(realty_le.classes_)}')
credit_app.head()

## 3. Feature / Target Split + Train/Test

In [ ]:
drop_cols    = ['ID', 'target']
feature_cols = [c for c in credit_app.columns if c not in drop_cols]

X = credit_app[feature_cols].fillna(0)
y = credit_app['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'X_train shape : {X_train.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'y_train dist  : {dict(y_train.value_counts())}')
print(f'y_test dist   : {dict(y_test.value_counts())}')

## 4. Helper — Confusion Matrix Heatmap

In [ ]:
def plot_cm(cm, model_name):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Predicted 0','Predicted 1'],
                yticklabels=['True 0','True 1'])
    ax.set_title(f'{model_name} — Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Model 1 — Logistic Regression

In [ ]:
def logistic_model(xtrain, xtest, ytrain, ytest):
    lr_model = LogisticRegression(random_state=42, max_iter=1000)
    lr_model.fit(xtrain, ytrain)
    y_pred = lr_model.predict(xtest)
    cm = confusion_matrix(ytest, y_pred)
    plot_cm(cm, 'Logistic Regression')
    print(classification_report(ytest, y_pred))
    return lr_model, y_pred

lr_model, lr_pred = logistic_model(X_train, X_test, y_train, y_test)

## 6. Model 2 — Random Forest

In [ ]:
def random_forest(X_train, X_test, y_train, y_test):
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    print('Random Forest Model Evaluation')
    cm = confusion_matrix(y_test, y_pred)
    plot_cm(cm, 'Random Forest')
    print(classification_report(y_test, y_pred))
    return rf_model, y_pred

rf_model, rf_pred = random_forest(X_train, X_test, y_train, y_test)

## 7. Model 3 — Decision Tree

In [ ]:
def d_tree(xtrain, xtest, ytrain, ytest):
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(xtrain, ytrain)
    ypred = dt.predict(xtest)
    print('*** DecisionTreeClassifier ***')
    print('Confusion matrix')
    cm = confusion_matrix(ytest, ypred)
    plot_cm(cm, 'Decision Tree')
    print('Classification report')
    print(classification_report(ytest, ypred))
    return dt, ypred

dt_model, dt_pred = d_tree(X_train, X_test, y_train, y_test)

## 8. Model 4 — XGBoost

In [ ]:
def xgboost_model(xtrain, xtest, ytrain, ytest):
    xgb = XGBClassifier(eval_metric='logloss', random_state=42,
                        use_label_encoder=False, verbosity=0)
    xgb.fit(xtrain, ytrain)
    ypred = xgb.predict(xtest)
    print('*** XGBoost Classifier ***')
    print('Confusion matrix')
    cm = confusion_matrix(ytest, ypred)
    plot_cm(cm, 'XGBoost')
    print('Classification report')
    print(classification_report(ytest, ypred))
    return xgb, ypred

xgb_model, xgb_pred = xgboost_model(X_train, X_test, y_train, y_test)

## 9. Model Comparison — F1-Score Table

In [ ]:
results = {
    'Logistic Regression': f1_score(y_test, lr_pred,  average='weighted'),
    'Random Forest'      : f1_score(y_test, rf_pred,  average='weighted'),
    'Decision Tree'      : f1_score(y_test, dt_pred,  average='weighted'),
    'XGBoost'            : f1_score(y_test, xgb_pred, average='weighted'),
}

results_df = pd.DataFrame(
    list(results.items()), columns=['Model','F1 Score (Weighted)']
).sort_values('F1 Score (Weighted)', ascending=False).reset_index(drop=True)

print('=== MODEL COMPARISON ===')
print(results_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(results_df['Model'], results_df['F1 Score (Weighted)'],
               color=['#8b5cf6','#3b82f6','#06b6d4','#10b981'])
ax.set_xlim(0, 1)
ax.set_xlabel('F1 Score (Weighted)')
ax.set_title('Model Comparison — F1 Score', fontweight='bold')
for bar, val in zip(bars, results_df['F1 Score (Weighted)']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 10. Hyperparameter Tuning — RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators'      : [50, 100, 200, 300],
    'max_depth'         : [None, 5, 10, 20, 30],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
    'max_features'      : ['sqrt', 'log2'],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
search = RandomizedSearchCV(
    rf_base, param_dist,
    n_iter=20, scoring='f1', cv=3,
    random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)

print(f'Best params : {search.best_params_}')
print(f'Best CV F1  : {search.best_score_:.4f}')

tuned_rf = search.best_estimator_
tuned_f1 = f1_score(y_test, tuned_rf.predict(X_test), average='weighted')
print(f'Tuned RF F1 on test set: {tuned_f1:.4f}')

## 11. Save Model & Encoders

In [ ]:
# Select best model
best_f1    = max(results.values())
best_model = tuned_rf if tuned_f1 >= best_f1 else rf_model
print(f'Final model chosen (F1={max(tuned_f1, best_f1):.4f}): {type(best_model).__name__}')

# Save model
with open('../model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('model.pkl saved ✓')

# Save encoders
encoders = {
    'gender_le'     : gender_le,
    'car_le'        : car_le,
    'realty_le'     : realty_le,
    'feature_cols'  : feature_cols,
    'housing_map'   : {'House / apartment':0,'Rented apartment':1,'With parents':2,
                       'Municipal apartment':3,'Co-op apartment':4,'Office apartment':5},
    'income_map'    : {'Working':0,'Commercial associate':1,'Pensioner':2,'State servant':3,'Student':4},
    'education_map' : {'Higher education':0,'Secondary / secondary special':1,
                       'Incomplete higher':2,'Lower secondary':3,'Academic degree':4},
    'family_map'    : {'Married':0,'Single / not married':1,'Civil marriage':2,'Separated':3,'Widow':4},
}

with open('../encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print('encoders.pkl saved ✓')

print('\n✅ Training complete! Run python app.py to start the Flask server.')